In [9]:
import os
os.chdir(r"C:\Users\imanb\Documents\motor-tpl-pricing-engine")

In [10]:
from pathlib import Path
import json
import pandas as pd

ART = Path("artifacts/reports")
ART.mkdir(parents=True, exist_ok=True)

FREQ = Path("data/staging/freq_staged.parquet")
SEV  = Path("data/staging/sev_staged.parquet")
SEV_TRAIN = Path("data/staging/sev_train.parquet")
SEV_UNMATCHED = Path("data/staging/sev_unmatched_claims.parquet")

In [11]:
from src.data.validate import validate_dataset, save_report
from src.data.schemas import FREQ_SCHEMA, SEV_SCHEMA

df_freq = pd.read_parquet(FREQ)
df_sev  = pd.read_parquet(SEV)

rep_freq = validate_dataset(df_freq, FREQ_SCHEMA, coerce=True, constraints_kwargs={"exposure_cap_policy": 1.0})
rep_sev  = validate_dataset(df_sev,  SEV_SCHEMA,  coerce=True)

save_report(rep_freq, ART / "freq_validation_report.json")
save_report(rep_sev,  ART / "sev_validation_report.json")

rep_freq.ok, rep_sev.ok, len(rep_freq.findings), len(rep_sev.findings)

(True, True, 1, 0)

In [12]:
staging_rep = json.loads(Path("artifacts/reports/staging_report.json").read_text(encoding="utf-8"))

# Exposure cap policy
freq_policies = staging_rep["policies"][0]["policies"]
cap = [p for p in freq_policies if p.get("policy") == "cap_exposure"][0]
idpol = [p for p in freq_policies if p.get("policy") == "canonicalize_idpol_to_int64"][0]

cap, idpol

({'policy': 'cap_exposure',
  'cap_value': 1.0,
  'rows_capped': 1224,
  'max_before': 2.01,
  'max_after': 1.0},
 {'policy': 'canonicalize_idpol_to_int64',
  'nulls': 0,
  'bad_parse': 0,
  'non_integer_like': 0,
  'dtype_after': 'int64'})

In [13]:
join_rep = json.loads(Path("artifacts/reports/sev_join_report.json").read_text(encoding="utf-8"))
join_rep["diagnostics"]

{'claims_rows_input': 26639,
 'rows_matched': 26444,
 'rows_unmatched': 195,
 'match_rate': 0.9926799054018545,
 'dup_policies_in_freq': 0,
 'missing_unique_idpol': 6,
 'top_missing_idpol_counts': {'2262511': 66,
  '2282134': 36,
  '2227533': 25,
  '2220367': 24,
  '2277846': 23,
  '2286775': 21},
 'merge_validate': 'many_to_one'}